# 05 — The REST API and the Python SDK

**What you will learn**
- The entity model: every asset is a typed entity with `id`, `fullyQualifiedName`, `version`, and relationships as entity references; one URL shape per type (`/v1/<type>s`, `/v1/<type>s/name/{fqn}`, `/v1/<type>s/{id}`).
- Reading well: `fields=` to ask for heavy parts, list + paging (`limit`, `after`/`before` cursors), `include=all|deleted|non-deleted`.
- Search over the API: `/v1/search/query` with `q`, `index`, `from`/`size`, and a query filter (Elasticsearch query JSON in `query_filter`).
- Changing safely: JSON Patch (`PATCH`, `application/json-patch+json`), PUT create-or-update, version history and `changeDescription`, soft delete vs hard delete and restore.
- The Python SDK (`metadata.ingestion.ometa.ometa_api.OpenMetadata`): typed `get_by_name`, `list_entities`, `patch_description`, and when it is worth it.

This chapter is Level 2 of the tutorial tree: it zooms into *"people and programs read and enrich it"* from Level 0, from the program side. The question it answers: *how do I read and change any entity from code, safely?* The running example stays Brightcup: every call below touches `raw.order_items` (shared, so only descriptions and one tier tag) or a throwaway table of this chapter's own.

*The cell below is the standard setup: find the project root from any working directory, put its `src/` on the path, and (re)load this chapter's helper module.*

In [1]:
import sys
from pathlib import Path

here = Path.cwd()
ROOT = next(d for d in [here, *here.parents, here / "project"] if (d / "pyproject.toml").exists() and (d / "src" / "openmetadata_tutorial").exists())
sys.path.insert(0, str(ROOT / "src"))

from openmetadata_tutorial import nb
ROOT = nb.setup()

import importlib
from openmetadata_tutorial import ch05_api as A
A = importlib.reload(A)  # a kernel started before the module changed holds the old copy otherwise

## The whole idea in one picture

Everything OpenMetadata knows — a table, a user, a tag, a test case — is an **entity**: a typed JSON document with a stable `id` (a UUID), a human-readable address (`fullyQualifiedName`, the FQN), and a `version` number. Related things (owners, tags, the database a table lives in) are stored as **entity references**: small `{"id", "type", "fullyQualifiedName"}` pointers, not copies. Programs talk to entities two ways: raw REST (any language) or the Python SDK (typed models over the same endpoints).

```mermaid
flowchart TB
    E["entity: a table"] --> I["id: stable UUID"]
    E --> F["fullyQualifiedName: dotted address"]
    E --> V["version + changeDescription"]
    E --> R["references: owners, tags, database"]
    C["GET /v1/tables/name/..."] --> E
    P["PATCH /v1/tables/{id}"] --> V
    S["GET /v1/search/query?q=..."] --> E
```

In plain words: **read** with GET (ask only for the parts you need), **find** with the search endpoint, **change** with PATCH (each change mints history), and **delete** in two stages (soft first, hard only when you mean it).

*Who does this chapter act as? The shared `client` helper authenticates with the ingestion-bot JWT from `project/.env` — the same token the ingestion workflows use.*

In [2]:
from openmetadata_tutorial import client
from openmetadata_tutorial.config import settings

s = settings()
print("API:", s.om_api)
print("acting as:", A.auth_info())

API: http://localhost:18585/api
acting as: {'user': 'ingestion-bot', 'isBot': True, 'via': 'OM_JWT_TOKEN in .env'}


> **Important:** Use a bot token, not the admin login, in automations.
> **What it means for us:** every change below is stamped `updatedBy: ingestion-bot`, so the audit trail shows *which automation* changed what; each automation gets its own bot whose token can be rotated independently.
> **Why it matters:** with a shared admin login, a bad script and a teammate look identical in the version history, and rotating the credential breaks everyone at once.

## Read: one GET, one entity

The simplest real example: fetch the order-lines table by its FQN. A **table** here means what chapter 01 called a data asset — here it is the raw `order_items` table (2,550 order lines over 30 products, from 1,000 orders; the next cell prints the row counts). The response is the entity: top-level keys, plus its `id`, `fullyQualifiedName`, `version` and `href` (its own URL).

In [3]:
t = client.get("/v1/tables/name/brightcup_pg.brightcup.raw.order_items")
print("keys:", sorted(t.keys()))
for k in ["id", "fullyQualifiedName", "version", "href", "name", "tableType"]:
    print(f"{k}: {t[k]}")
from openmetadata_tutorial import shop_data
print("shop rows — order_items:", len(shop_data.load("order_items")), "| orders:", len(shop_data.load("orders")))

keys: ['changeDescription', 'columns', 'database', 'databaseSchema', 'deleted', 'description', 'entityStatus', 'fullyQualifiedName', 'href', 'id', 'incrementalChangeDescription', 'lifeCycle', 'name', 'processedLineage', 'service', 'serviceType', 'sourceHash', 'tableType', 'updatedAt', 'updatedBy', 'version']
id: 63ae2937-3885-4dc5-969f-4597bec08615
fullyQualifiedName: brightcup_pg.brightcup.raw.order_items
version: 0.3
href: http://localhost:18585/api/v1/tables/63ae2937-3885-4dc5-969f-4597bec08615
name: order_items
tableType: Regular
shop rows — order_items: 2550 | orders: 1000


*What it means for us:* the catalog address of this table has three parts — service `brightcup_pg`, database `brightcup`, schema `raw` — joined by dots, and the entity carries both a machine id and that human address. Every other entity type (users, glossaries, test cases) has the same envelope: `id`, `name`, `fullyQualifiedName`, `version`, `href`.

*Heavy parts are opt-in — with one exception already visible: cell 8's key list includes `columns`, so the default GET already ships the column list. Re-fetch the same table asking for `owners` and `tags` explicitly, and measure both payloads in characters. For contrast, do the same on `raw.orders`, which other chapters already annotated with an owner and tags, asking for five extras (`owners,tags,followers,domains,votes`).*

In [4]:
plain = A.get_table()
full = A.get_table(fields="owners,tags")
print(f"order_items — plain: {A.payload_size(plain)} chars; with owners,tags: {A.payload_size(full)} chars")
o_plain = client.get("/v1/tables/name/brightcup_pg.brightcup.raw.orders")
o_full = client.get("/v1/tables/name/brightcup_pg.brightcup.raw.orders", fields="owners,tags,followers,domains,votes")
print(f"raw.orders — plain: {A.payload_size(o_plain)} chars; with 5 extras: {A.payload_size(o_full)} chars")

order_items — plain: 3734 chars; with owners,tags: 4251 chars
raw.orders — plain: 3866 chars; with 5 extras: 5507 chars


> **Important:** Ask only for the fields you need.
> **What it means for us:** pass `fields=` with exactly the heavy parts the task needs (`owners`, `tags`, `followers`, `domains`, `votes`); the default GET already includes `columns`, but leaves the rest out.
> **Why it matters:** `fields=*`-style fetching on big tables with many columns and tags is slow and heavy — here the annotated `raw.orders` grew from 3866 to 5507 chars (about 42%) just from asking for five extras.

*What it means for us, concretely:* on the small, unannotated `order_items` the two extra fields add about 500 characters (3734 to 4251); on the annotated `raw.orders` the five extras add about 1,600. `fields` is the main performance knob of the read API — and its limit is that you must know the field names upfront (unknown names return a 400, as the errors section shows).

## Same entity, three addresses

One entity, three URLs: by name (`/v1/tables/name/{fqn}`), by id (`/v1/tables/{id}`), and the `href` the server hands back (a full URL to the by-id address). All three return the same `id` below. (One FQN rule for the road: a name part that itself contains a dot is quoted, `"a.b"` — no Brightcup name needs it, but hand-built FQNs do.)

In [5]:
import requests

by_name = client.get(f"/v1/tables/name/{A.ORDER_ITEMS}")
by_id = client.get(f"/v1/tables/{by_name['id']}")
r = requests.get(by_name["href"], headers={"Authorization": f"Bearer {client.token()}"}, timeout=30)
r.raise_for_status()
print("by name, by id, via href — same entity:", by_name["id"] == by_id["id"] == r.json()["id"])
print("id:", by_name["id"])

by name, by id, via href — same entity: True
id: 63ae2937-3885-4dc5-969f-4597bec08615


*What it means for us:* store the `id` when a program tracks an entity across calls (it never changes), and use the FQN when a person reads the code (it says where the asset lives). `href` saves clients from assembling URLs — follow it.

## List and page: the whole database, three at a time

List endpoints return a page (`data`) plus a `paging` object with an opaque `after` cursor and the `total`. Ask for 3 of the 8 tables/views: you get the first three alphabetically and a cursor for the rest. (`total` counts everything in database `brightcup_pg.brightcup`; the sandbox service from chapter 04 lives in a different database and is not counted.)

In [6]:
page = client.get("/v1/tables", database="brightcup_pg.brightcup", limit=3)
print("first page:", [d["fullyQualifiedName"] for d in page["data"]])
print("total:", page["paging"].get("total"), "| more pages:", bool(page["paging"].get("after")))

first page: ['brightcup_pg.brightcup.analytics.ch06_dynamic', 'brightcup_pg.brightcup.analytics.customer_ltv', 'brightcup_pg.brightcup.raw.customers']
total: 8 | more pages: True


*Follow the cursors until no `after` comes back — the loop lives in `A.list_all`, a generator, so cells never juggle cursors by hand.*

In [7]:
all_tables = list(A.list_all("/v1/tables", database="brightcup_pg.brightcup", limit=3))
print("walked all pages:", len(all_tables), "tables/views")
for e in all_tables:
    print(" -", e["fullyQualifiedName"])

walked all pages: 8 tables/views
 - brightcup_pg.brightcup.analytics.ch06_dynamic
 - brightcup_pg.brightcup.analytics.customer_ltv
 - brightcup_pg.brightcup.raw.customers
 - brightcup_pg.brightcup.analytics.daily_revenue
 - brightcup_pg.brightcup.analytics.fct_orders
 - brightcup_pg.brightcup.raw.order_items
 - brightcup_pg.brightcup.raw.orders
 - brightcup_pg.brightcup.raw.products


*What it means for us:* 8 assets walked three-at-a-time — the 7 Brightcup seed tables/views from the plan plus `analytics.ch06_dynamic`, the extra view chapter 06 creates. The loop is idempotent and complete: when `after` disappears, everything has been seen. Its limit: paging order and totals reflect the catalog *right now* — another chapter's new asset shows up in your next walk (sharing rule, not a bug).

*Cursors go both ways: every page after the first also carries a `before` cursor that steps back to the previous page. The cell below walks one step forward with `after`, then steps back with `before` — landing on the exact first page again.*

In [8]:
p1 = client.get("/v1/tables", database="brightcup_pg.brightcup", limit=3)
p2 = client.get("/v1/tables", database="brightcup_pg.brightcup", limit=3, after=p1["paging"]["after"])
back = client.get("/v1/tables", database="brightcup_pg.brightcup", limit=3, before=p2["paging"]["before"])
print("page 2:", [d["name"] for d in p2["data"]])
print("first page has a before cursor:", bool(p1["paging"].get("before")))
print("back via before == page 1:", [d["id"] for d in back["data"]] == [d["id"] for d in p1["data"]])

page 2: ['daily_revenue', 'fct_orders', 'order_items']
first page has a before cursor: False
back via before == page 1: True


## Search: text query, then a filter

The search endpoint (`/v1/search/query`) ranks by text relevance over the Elasticsearch index — it finds things by words, not by exact address. Ask for `revenue` on the table index: one hit, the finance view this whole tutorial is about.

In [9]:
hits = A.search_frame("revenue", index="table_search_index")
print("total:", hits.attrs["total"])
print(hits.to_string())

total: 1
  entity                                             fqn  kind                                                   description
0  table  brightcup_pg.brightcup.analytics.daily_revenue  View  Revenue and order count per day. Feeds the finance dashboard


*What it means for us:* one query, one answer — `daily_revenue` is the only table with "revenue" in its name or description. Now the same query restricted to the `analytics` schema with an Elasticsearch bool filter passed as `query_filter`, plus a broader word (`orders`, 6 hits) to show the filter narrowing to 4.

In [10]:
print("revenue in analytics:", A.search_frame("revenue", query_filter=A.analytics_only_filter()).attrs["total"])
print("orders, all schemas:", A.search_frame("orders").attrs["total"])
framed = A.search_frame("orders", query_filter=A.analytics_only_filter())
print("orders in analytics:", framed.attrs["total"])
print(framed[["fqn"]].to_string())

revenue in analytics: 1


orders, all schemas: 6


orders in analytics: 4
                                              fqn
0     brightcup_pg.brightcup.analytics.fct_orders
1  brightcup_pg.brightcup.analytics.daily_revenue
2   brightcup_pg.brightcup.analytics.customer_ltv
3   brightcup_pg.brightcup.analytics.ch06_dynamic


*Search more than tables: `index=all` spans every entity type — columns, test cases, glossary terms, dashboards, data products, tags — in one call. The default page holds 10 hits, so watch the `total`:*

In [11]:
wide = A.search_frame("revenue", index="all")
print("total:", wide.attrs["total"])
print(wide[["entity", "fqn"]].to_string())

total: 11
         entity                                                                    fqn
0   tableColumn                 brightcup_pg.brightcup.analytics.daily_revenue.revenue
1      testCase  brightcup_pg.brightcup.analytics.daily_revenue.daily_revenue_has_rows
2  glossaryTerm                                                      Brightcup.Revenue
3         table                         brightcup_pg.brightcup.analytics.daily_revenue
4     testSuite                                   231b70a4-3f84-4215-9b09-cef033ff9a98
5     testSuite               brightcup_pg.brightcup.analytics.daily_revenue.testSuite
6   dataProduct                                                      Revenue Reporting
7         chart                                      brightcup_metabase.revenue_by_day
8  glossaryTerm                                               Brightcup.Lifetime value
9     dashboard                                    brightcup_metabase.finance_overview


*What it means for us:* the catalog is searchable as one pool — the `revenue` idea links a table, its column, its quality tests, the `Brightcup.Revenue` glossary term, the `Revenue Reporting` data product and the finance dashboard. And note the gap the output shows: `total` says 11 but the table lists 10 rows — the default `size` is 10, so one hit (the `Tier.Tier1` tag, as the next cell shows) is cut off. Its limit: search matches words, it does not judge relevance to your task — `query_filter` (any Elasticsearch bool query, here "schema is analytics") is how a program narrows to what it may act on.

*Page through search hits with `from`/`size`, the search-side equivalent of the list cursors: three slices (0–5, 5–10, 10–15) cover all 11 hits with no overlap, and the last slice holds the one hit the default page cut off.*

In [12]:
pg1 = A.search_frame("revenue", index="all", size=5, frm=0)
pg2 = A.search_frame("revenue", index="all", size=5, frm=5)
pg3 = A.search_frame("revenue", index="all", size=5, frm=10)
print("total:", pg1.attrs["total"], "| pages:", len(pg1), len(pg2), len(pg3))
print("page 3 (the cut-off hit):", pg3[["entity", "fqn"]].to_string())
print("no overlap:", not (set(pg1["fqn"]) & set(pg2["fqn"])) and not (set(pg1["fqn"]) & set(pg3["fqn"])) and not (set(pg2["fqn"]) & set(pg3["fqn"])))

total: 11 | pages: 5 5 1
page 3 (the cut-off hit):   entity         fqn
0    tag  Tier.Tier1
no overlap: True


## Change with JSON Patch (medium)

Reads were the simple part; writes are the medium part, with one new idea: **JSON Patch**. A change is a list of `add`/`replace`/`remove` operations on paths (`/description`, `/columns/2/description`), sent with `PATCH` and content type `application/json-patch+json` (the `client` helper sets it). Contrast with PUT: PUT sends a whole entity and creates-or-updates it (what chapter 04's ingestion uses, and what the scratch table below is built with); PATCH is a surgical edit to one existing entity. Two limits up front, both visible in the cells below: column paths are index-based (the callout says how to handle that); and a patch that changes nothing returns the entity with the *same* version — no new version, no history entry. Only `raw.order_items` is edited here — a shared table, so descriptions and one tier tag only. (To watch a patch mint a new version live in a single run, see the scratch table below: there one description patch steps a fresh 0.1 table to 0.2.)

First, the table description (the helper uses `op: add`, which overwrites when a description already exists — and the cell patches twice, so the no-op is visible live whatever version the table is at):

In [13]:
eid = A.get_table()["id"]
first = A.set_description(eid, A.TABLE_DESCRIPTION)
second = A.set_description(eid, A.TABLE_DESCRIPTION)
print("description:", second.get("description"))
print("first PATCH version:", first.get("version"), "| second PATCH version:", second.get("version"),
      "| same:", first.get("version") == second.get("version"))

description: One row per ordered product: which order, which product, how many, at what price.
first PATCH version: 0.3 | second PATCH version: 0.3 | same: True


> **Important:** JSON Patch column paths are index-based (`/columns/2/description`).
> **What it means for us:** look up the index by name *right before* patching (`A.column_index`), in the same breath as the patch — never hardcode a number.
> **Why it matters:** a re-ingestion can reorder or add columns, and a stale index silently edits the wrong column. The `400` in the errors section is what a bad path looks like.

*The column description for `quantity` — index resolved by name, then patched:*

In [14]:
table = A.get_table(fields="columns")
i = A.column_index(table, "quantity")
print("quantity is column index:", i)
first = A.set_column_description(table["id"], "quantity", A.QUANTITY_DESCRIPTION)
second = A.set_column_description(table["id"], "quantity", A.QUANTITY_DESCRIPTION)
print("quantity description:", second["columns"][i].get("description"))
print("first PATCH version:", first.get("version"), "| second PATCH version:", second.get("version"),
      "| same:", first.get("version") == second.get("version"))

quantity is column index: 2


quantity description: Number of units of the product in this order line.
first PATCH version: 0.3 | second PATCH version: 0.3 | same: True


*What it means for us:* the edit applied — the printed description matches — and both patches print the same version: the second send changed nothing, so the server minted no new version. Re-running this cell is safe at any version, on any stack. The versions table in the next section shows the history these edits left when they first landed. Next, the tier tag, appended to `/tags/-` with the exact tag-label shape the server returns (verified against the `Tier.Tier2` tag on `raw.orders`).*

In [15]:
eid = A.get_table()["id"]
first = A.ensure_tier_tag(eid)
second = A.ensure_tier_tag(eid)
tags = [g["tagFQN"] for g in A.get_table(fields="tags").get("tags", [])]
print("tags:", tags, "| Tier.Tier3 count:", tags.count("Tier.Tier3"))
print("first PATCH version:", first.get("version"), "| second PATCH version:", second.get("version"),
      "| same:", first.get("version") == second.get("version"))

tags: ['Tier.Tier3'] | Tier.Tier3 count: 1
first PATCH version: 0.3 | second PATCH version: 0.3 | same: True


*What it means for us:* `Tier.Tier3` is on the table, exactly once — re-running the cell does not duplicate it (the server dedupes identical tag labels), and the version does not move for the same reason as above (both patches print the same version). All three patches are idempotent: same text, same tag, same version — the notebook converges instead of accumulating. (Note the cell reads tags back with `fields="tags"`: the plain GET leaves them out, as the read section showed.)*

## Versions: the audit log of an entity

Every change leaves a version: `GET /v1/tables/{id}/versions` lists them newest-first, and each entry carries a `changeDescription` — which fields were added/updated/deleted and the previous version. One row per version, oldest last:

In [16]:
t = A.get_table()
frame = A.versions_frame(t["id"])
print(frame.to_string())
d = A.change_detail(t)
print("current version:", t.get("version"), "| previousVersion:", d["previousVersion"], "|", d["summary"])

   version      updatedBy               updatedAt  previousVersion                          changed
0      0.3  ingestion-bot 2026-10-12 11:47:20.236              0.2  added:tags; updated:description
1      0.2  ingestion-bot 2026-10-12 10:58:17.027              0.1           added:tableConstraints
2      0.1  ingestion-bot 2026-10-12 10:57:37.770              NaN                                 
current version: 0.3 | previousVersion: 0.2 | added:tags; updated:description


*What it means for us:* this is the audit trail the UI's version history (chapter 02) renders — newest first, one row per version, each saying which fields changed and which version came before (`previousVersion`; the ingested first version has none). In the run saved here the top row carries this chapter's edits — the tier tag plus the table description (`added:tags; updated:description`), all stamped `ingestion-bot` — and the `previousVersion` chain leads back to the ingested 0.1 through a middle row that added the foreign-key constraint. The next cell checks one subtlety live: what the stored snapshots say about the `quantity` column versus what each version's own `changeDescription` lists. When something looks wrong on a table, this table of versions is the first place to look, before asking who ran what.*

In [17]:
frame = A.versions_frame(A.get_table()["id"])
for v in frame["version"].tolist():
    s = A.snapshot_column(A.get_table()["id"], str(v), "quantity")
    print(f"version {v}: quantity description set: {s['description'] == A.QUANTITY_DESCRIPTION} | change: {s['summary'] or '(none)'}")

version 0.3: quantity description set: True | change: added:tags; updated:description
version 0.2: quantity description set: True | change: added:tableConstraints


version 0.1: quantity description set: False | change: (none)


*The 0.2 and 0.3 snapshots already carry the `quantity` description — the 0.1 snapshot does not, so the edit landed between 0.1 and 0.2 — yet no version's `changeDescription` mentions columns (0.2 lists only the added table constraint). The nested column edit rode along on the version without its own history entry. So `changeDescription` is a reliable log of top-level field changes, not of every nested touch — to audit a column, read the version snapshots, not just the change summaries.*

## Errors you will meet

Four real ones, raised as `requests.HTTPError` with a JSON body (`code`, `message`). The helper maps each to a status plus message; the cell just prints them:

In [18]:
for kind in ["missing", "bad-patch", "bad-fields", "bad-token"]:
    e = A.probe_error(kind)
    print(kind, "->", e["status"], "|", e["message"])

missing -> 404 | table instance for brightcup_pg.brightcup.raw.NOPE not found


bad-patch -> 400 | Invalid field 'nope' for Table. Valid fields are: aliases, certification, changeDescription, children, columns, compress
bad-fields -> 400 | Invalid field name BOGUS_FIELD


bad-token -> 401 | Not Authorized! Invalid token.


- **404** — the FQN does not exist (typo, or the asset was hard-deleted): check the address.
- **400** — the request itself is wrong (here: patch path `/nope` is not a table field): read the message, it lists the valid fields.
- **400** — same status, different mistake: `fields=BOGUS_FIELD` is not a field the API knows (unknown names return a 400, as the read section warned).
- **401** — the token is missing, expired or bogus (`Bearer x`): refresh the bot token (`just token` writes it to `.env`).

*What it means for us:* always print `status + message` when a call fails — OpenMetadata's messages name the exact problem (unknown instance, invalid field, invalid token), so there is rarely a need to guess.

## Delete and restore (complex): a throwaway entity, end to end

The complex round trip runs on `raw.ch05_scratch`, a table that exists **only in the catalog** — created below by PUT, never in Postgres. (PUT is create-or-update: `201` on first creation, `200` when the table already exists; running this cell twice keeps one table. And a heads-up from chapter 04: the next metadata re-run with `markDeletedTables: true` will soft-delete this catalog-only table on sight — that is expected, not breakage.)

In [19]:
scratch = A.ensure_scratch()
print("PUT status:", scratch["status"], "| fqn:", scratch["fqn"], "|", scratch["how"])
print("fresh version:", client.get(f"/v1/tables/{scratch['id']}").get("version"))

PUT status: 201 | fqn: brightcup_pg.brightcup.raw.ch05_scratch | PUT create-or-update
fresh version: 0.1


*A fresh entity is where `patch → new version` shows live: one description patch on the 0.1 scratch table mints 0.2 with its own `changeDescription` entry — the same 0.1 step the shared table's history shows, but visible in a single run because nothing else has touched this entity.*

In [20]:
t = A.patch(f"/v1/tables/{scratch['id']}", [{"op": "add", "path": "/description", "value": "Throwaway table for the chapter 05 delete/restore demo."}])
print("after PATCH: version", t.get("version"), "|", A.change_summary(t))

after PATCH: version 0.2 | added:description


> **Important:** Soft delete is the default; hard delete is final.
> **What it means for us:** plain `DELETE` hides the entity (plain GET goes 404) but keeps it recoverable via `include=deleted` + restore; `hardDelete=true&recursive=true` also takes its versions, lineage edges and test cases with no undo.
> **Why it matters:** hard-deleting a real table destroys history and breaks every downstream edge — prototype deletions on scratch entities like this one, never on shared assets.

*Soft-delete it: gone from normal reads and from `include=non-deleted`, visible with `include=deleted`:*

In [21]:
print("DELETE status:", client.api("DELETE", f"/v1/tables/{scratch['id']}").status_code)
print("plain GET:", A.get_status(scratch["id"]))
print("include=non-deleted:", A.get_status(scratch["id"], include="non-deleted"))
print("include=deleted:", A.get_status(scratch["id"], include="deleted"))

DELETE status: 200
plain GET: {'status': 404, 'deleted': None, 'name': None}


include=non-deleted: {'status': 404, 'deleted': None, 'name': None}
include=deleted: {'status': 200, 'deleted': True, 'name': 'ch05_scratch'}


*Restore it, then hard-delete it: even `include=all` goes 404. Each step prints its status:*

In [22]:
print("restore status:", client.api("PUT", "/v1/tables/restore", json={"id": scratch["id"]}).status_code,
      "| after restore:", A.get_status(scratch["id"]))
print("hard DELETE status:", client.api("DELETE", f"/v1/tables/{scratch['id']}?hardDelete=true&recursive=true").status_code)
print("include=all after hard delete:", A.get_status(scratch["id"], include="all"))

restore status: 200 | after restore: {'status': 200, 'deleted': False, 'name': 'ch05_scratch'}


hard DELETE status: 200
include=all after hard delete: {'status': 404, 'deleted': None, 'name': None}


*What it means for us:* the full lifecycle in four calls — create-or-update, soft delete (recoverable), restore, hard delete (final). Re-running the notebook recreates the scratch table and walks the same cycle, ending clean: nothing of this chapter's stays behind except descriptions and one tag on `order_items`.

## The Python SDK: the same API with typed models

`client.sdk()` returns the official `OpenMetadata` client (`metadata.ingestion.ometa.ometa_api.OpenMetadata`, same version as the server, 2.0.5). Same entity, now as a pydantic model instead of a dict — `t.columns[0].dataType` is a real enum, and names and descriptions are thin wrappers (hence `.root` in the cell): the price of typing is paid once, the attribute access works forever:

In [23]:
from metadata.generated.schema.entity.data.table import Table

md = client.sdk()
t = md.get_by_name(entity=Table, fqn=A.ORDER_ITEMS, fields=["columns"])
print("model:", type(t).__name__)
print("first column:", t.columns[0].name.root, "|", t.columns[0].dataType)
print("description:", t.description.root)

model: Table
first column: order_id | DataType.INT
description: One row per ordered product: which order, which product, how many, at what price.


*List through the SDK: one call returned all 8 entities with no cursor left — it walked the pages itself:*

In [24]:
lst = md.list_entities(entity=Table, params={"database": "brightcup_pg.brightcup", "limit": 3})
print("entities:", len(lst.entities), "| total:", lst.total, "| cursor left:", lst.after)

entities: 8 | total: 8 | cursor left: None


*Write through the SDK on `raw.order_items`, setting the same text as the REST step so state stays stable. Its limit: `patch_description` never overwrites blindly — without `force` it protects an existing description and returns `None`, and with `force=True` but identical text it also returns `None`, patching only on a real difference, so the version does not move.*

In [25]:
t = md.get_by_name(entity=Table, fqn=A.ORDER_ITEMS, fields=["columns"])
before = A.get_table()["version"]
print("no force (description exists):", md.patch_description(entity=Table, source=t, description=A.TABLE_DESCRIPTION))
t = md.get_by_name(entity=Table, fqn=A.ORDER_ITEMS, fields=["columns"])
print("force, same text:", md.patch_description(entity=Table, source=t, description=A.TABLE_DESCRIPTION, force=True))
print("REST version before/after:", before, A.get_table()["version"])

no force (description exists): None
force, same text: None
REST version before/after: 0.3 0.3


*How heavy is the SDK? One measured fact: the installed ingestion distribution and the server version, which must match (the client is pinned to the server) — against REST, which needs only `requests`:*

In [26]:
rel = A.sdk_release()
print("SDK dist:", rel["pkg"], "| server:", rel["server"], "| match:", rel["pkg"].startswith(rel["server"]))

SDK dist: 2.0.5.0 | server: 2.0.5 | match: True


## REST vs SDK: which to use

| concern | REST (`client.get` / `client.api`) | SDK (`client.sdk()`) |
|---|---|---|
| what you get | plain JSON dicts (`t["columns"][2]["description"]`), any language | pydantic models with attribute access (`t.columns[0].dataType` is a `DataType` enum, names/descriptions are `.root` wrappers — see the cell above) |
| auth | `Authorization: Bearer` header per call | connection object once, then method calls |
| paging | follow `after`/`before` cursors yourself (`A.list_all`) | list helpers walk the pages for you (`after` comes back `None`) |
| versions | the same server-side numbers both clients read (0.3 above); a no-op PATCH returns the same version with no new history entry | the same numbers; `patch_description` returns `None` instead of sending a no-op patch, so the version never moves spuriously |
| weight | only `requests` | the `openmetadata-ingestion` distribution, pinned to the server version (2.0.5.0 vs server 2.0.5, printed above) |

*Recommendation: the SDK inside Python automations (typing pays off the moment more than one entity type is involved), raw REST from anything else — and from small scripts where the ingestion dependency is not worth it.*

## Carry forward

- **One API shape for all entity types**: by name, by id, list with paging, `fields`, `include` — learn it on tables, reuse it on glossaries, users, test cases.
- **Changes are JSON Patch → new version with `changeDescription`**; that is the audit trail. Column paths are index-based (resolve by name first), and soft delete is the default because hard delete takes history with it.
- **The SDK is the same API with typed models**; choose by language and need — SDK for Python automations, REST for everything else. Either way, automate as a bot so `updatedBy` says which program changed what.